# 🔐 Quantum Secure Payment Authentication
## BB84 Quantum Key Distribution + Quantum ML Fraud Detection

**A complete end-to-end secure payment system combining:**
- 🔑 BB84 Quantum Key Distribution (Qiskit)
- 🤖 Variational Quantum Classifier for Fraud Detection (PennyLane)
- ⚛️ Quantum Circuit Visualization (Cirq)
- 🌐 Interactive Web UI (Gradio)
- 🛡️ Secure Transaction Simulation




In [1]:
# ============================================================
# CELL 1 – Install Dependencies
# ============================================================
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

print('📦 Installing quantum computing frameworks...')
install('pennylane')
install('pennylane-qiskit')
install('qiskit')
install('qiskit-aer')
install('cirq')

print('📦 Installing ML and data science libraries...')
install('scikit-learn')
install('imbalanced-learn')
install('pandas')
install('numpy')
install('matplotlib')
install('plotly')

print('📦 Installing web interface...')
install('gradio')

print('✅ All dependencies installed!')


📦 Installing quantum computing frameworks...
📦 Installing ML and data science libraries...
📦 Installing web interface...
✅ All dependencies installed!


In [2]:
# ============================================================
# CELL 2 – Import Libraries
# ============================================================
import os, sys, json, time, random, hashlib, warnings, io, base64
from datetime import datetime
from typing import List, Tuple, Dict, Optional
from collections import Counter
warnings.filterwarnings('ignore')

# Data science
import numpy as np
import pandas as pd

# Visualization
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px

# Classical ML
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, accuracy_score,
                              roc_auc_score, roc_curve, confusion_matrix)
from imblearn.over_sampling import SMOTE

# PennyLane (Quantum ML)
import pennylane as qml
from pennylane import numpy as pnp

# Qiskit (BB84)
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

# Cirq
import cirq

# Gradio UI
import gradio as gr

from IPython.display import display, HTML

print('✅ Imports successful!')
print(f'  PennyLane : {qml.__version__}')
print(f'  Cirq      : {cirq.__version__}')
print(f'  Gradio    : {gr.__version__}')


✅ Imports successful!
  PennyLane : 0.44.0
  Cirq      : 1.6.1
  Gradio    : 5.50.0


In [3]:
# ============================================================
# CELL 3 – Dataset Loading
# ============================================================
# Uses real Credit Card Fraud dataset if available (upload creditcard.csv
# to /content/), otherwise generates a realistic synthetic dataset.

def generate_synthetic_fraud_data(n_samples=10_000, fraud_ratio=0.02, seed=42):
    """Generate synthetic fraud dataset matching Kaggle statistics."""
    rng = np.random.RandomState(seed)
    n_fraud  = int(n_samples * fraud_ratio)
    n_normal = n_samples - n_fraud
    rows = []

    for _ in range(n_normal):   # Normal transactions
        row = {'Time': rng.uniform(0, 172800), 'Amount': rng.exponential(88)}
        for i in range(1, 29): row[f'V{i}'] = rng.normal(0, 1)
        row['Class'] = 0
        rows.append(row)

    for _ in range(n_fraud):    # Fraudulent transactions (shifted features)
        row = {'Time': rng.uniform(0, 172800), 'Amount': rng.exponential(122)}
        for i in range(1, 29):
            shift = rng.choice([-3, -2, 2, 3]) if i in [3,4,9,10,11,12,14,16] else 0
            row[f'V{i}'] = rng.normal(shift, 1.5)
        row['Class'] = 1
        rows.append(row)

    return pd.DataFrame(rows).sample(frac=1, random_state=seed).reset_index(drop=True)


def load_dataset():
    for path in ['creditcard.csv', '/content/creditcard.csv']:
        if os.path.exists(path):
            print(f'📂 Loading real dataset from: {path}')
            df = pd.read_csv(path)
            print(f'  Rows: {len(df):,}  Fraud: {df["Class"].sum():,}')
            return df
    print('⚠️  Real dataset not found → using synthetic dataset')
    df = generate_synthetic_fraud_data()
    print(f'  Synthetic rows: {len(df):,}  Fraud: {df["Class"].sum():,}')
    return df

df_raw = load_dataset()
print(f'\nClass distribution:\n{df_raw["Class"].value_counts()}')
display(df_raw.head(3))


⚠️  Real dataset not found → using synthetic dataset
  Synthetic rows: 10,000  Fraud: 200

Class distribution:
Class
0    9800
1     200
Name: count, dtype: int64


,Time,Amount,V1,V2,V3,V4,V5,V6,V7,V8,...,V20,V21,V22,V23,V24,V25,V26,V27,V28,Class
0,87285.809306,7.538700,-1.486698,-0.208409,-0.979168,-2.604310,0.323700,-2.779626,-1.019828,0.750055,...,0.720957,0.813683,-0.296312,-1.004872,-1.855971,1.216479,-0.395923,-0.561117,0.076069,0
1,172085.450324,23.347266,-0.613132,0.456177,1.092686,-1.823715,-0.609361,-1.219153,-0.344511,0.610155,...,-0.777654,-0.102999,0.746603,1.523011,0.352478,0.202785,0.075930,0.858158,0.777044,0
2,20283.212165,107.915593,-1.166690,0.761554,0.194845,0.655836,-1.710793,1.832643,0.252280,-1.807232,...,0.120860,0.331094,-1.470422,-0.830373,0.085013,0.185705,0.400707,1.406046,-0.094209,0


In [4]:
# ============================================================
# CELL 4 – Data Preprocessing
# ============================================================

def preprocess_data(df, n_quantum_features=4, test_size=0.2, seed=42):
    df = df.copy()
    # Scale Amount and Time
    scaler_a = StandardScaler()
    scaler_t = StandardScaler()
    df['Amount_scaled'] = scaler_a.fit_transform(df[['Amount']])
    df['Time_scaled']   = scaler_t.fit_transform(df[['Time']])
    df.drop(['Amount', 'Time'], axis=1, inplace=True)

    X = df.drop('Class', axis=1).values
    y = df['Class'].values
    feature_names = list(df.drop('Class', axis=1).columns)

    # Train / test split (stratified)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=seed, stratify=y)

    # SMOTE oversampling on training set
    smote = SMOTE(random_state=seed)
    X_train_r, y_train_r = smote.fit_resample(X_train, y_train)

    # Quantum feature subset: V3, V4, V9, V10 (most discriminative)
    q_cols = [feature_names.index(c) for c in ['V3','V4','V9','V10']
              if c in feature_names]
    if len(q_cols) < n_quantum_features:
        q_cols = list(range(n_quantum_features))

    # Scale to [0, pi] for angle encoding
    q_scaler = MinMaxScaler(feature_range=(0, np.pi))
    X_q_train = q_scaler.fit_transform(X_train_r[:, q_cols])
    X_q_test  = q_scaler.transform(X_test[:, q_cols])

    print(f'✅ Preprocessing complete')
    print(f'  Train (balanced): {X_train_r.shape}  Fraud: {y_train_r.sum():,}')
    print(f'  Test            : {X_test.shape}    Fraud: {y_test.sum():,}')
    print(f'  Quantum features: {q_cols}')
    return X_train_r, X_test, y_train_r, y_test, X_q_train, X_q_test, q_scaler, q_cols, feature_names


(X_train, X_test, y_train, y_test,
 X_q_train, X_q_test,
 q_scaler, q_cols, feature_names) = preprocess_data(df_raw)


✅ Preprocessing complete
  Train (balanced): (15680, 30)  Fraud: 7,840
  Test            : (2000, 30)    Fraud: 40
  Quantum features: [2, 3, 8, 9]


In [5]:
# ============================================================
# CELL 5 – Classical Fraud Detection Baseline
# ============================================================

class ClassicalFraudDetector:
    """Logistic Regression + Random Forest ensemble."""

    def __init__(self):
        self.lr     = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
        self.rf     = RandomForestClassifier(n_estimators=100, random_state=42,
                                             class_weight='balanced', n_jobs=-1)
        self.scaler = StandardScaler()
        self.metrics: Dict[str, Dict] = {}

    def fit(self, X, y):
        Xs = self.scaler.fit_transform(X)
        print('🏃 Training Logistic Regression... ', end=''); self.lr.fit(Xs, y);  print('done')
        print('🌲 Training Random Forest...       ', end=''); self.rf.fit(Xs, y);  print('done')

    def _eval(self, name, model, X_test, y_test):
        Xs    = self.scaler.transform(X_test)
        y_hat = model.predict(Xs)
        y_pr  = model.predict_proba(Xs)[:, 1]
        self.metrics[name] = {
            'accuracy': accuracy_score(y_test, y_hat),
            'auc'     : roc_auc_score(y_test, y_pr),
            'y_pred'  : y_hat, 'y_prob': y_pr,
            'report'  : classification_report(y_test, y_hat,
                                              target_names=['Normal','Fraud'])
        }
        print(f'  {name}: Acc={self.metrics[name]["accuracy"]:.4f}  AUC={self.metrics[name]["auc"]:.4f}')

    def evaluate(self, X_test, y_test):
        print('\n📊 Classical Model Evaluation:')
        self._eval('Logistic Regression', self.lr, X_test, y_test)
        self._eval('Random Forest',       self.rf, X_test, y_test)

    def predict_proba_rf(self, x):
        Xs = self.scaler.transform(x.reshape(1, -1))
        return float(self.rf.predict_proba(Xs)[0, 1])

    def predict_proba_lr(self, x):
        Xs = self.scaler.transform(x.reshape(1, -1))
        return float(self.lr.predict_proba(Xs)[0, 1])

    def plot_roc(self):
        fig, ax = plt.subplots(figsize=(7, 5))
        for name, color in [('Logistic Regression','royalblue'), ('Random Forest','darkorange')]:
            if name in self.metrics:
                fpr, tpr, _ = roc_curve(y_test, self.metrics[name]['y_prob'])
                ax.plot(fpr, tpr, color=color, lw=2,
                        label=f'{name} (AUC={self.metrics[name]["auc"]:.3f})')
        ax.plot([0,1],[0,1],'k--',lw=1)
        ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
        ax.set_title('ROC Curves – Classical Baseline')
        ax.legend(); ax.grid(alpha=0.3); plt.tight_layout()
        return fig


classical_detector = ClassicalFraudDetector()
classical_detector.fit(X_train, y_train)
classical_detector.evaluate(X_test, y_test)
print('\n📋 Random Forest Report:')
print(classical_detector.metrics['Random Forest']['report'])
classical_detector.plot_roc()
plt.show()


🏃 Training Logistic Regression... done
🌲 Training Random Forest...       done

📊 Classical Model Evaluation:
  Logistic Regression: Acc=0.6535  AUC=0.6018
  Random Forest: Acc=0.9955  AUC=0.9990

📋 Random Forest Report:
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00      1960
       Fraud       0.92      0.85      0.88        40

    accuracy                           1.00      2000
   macro avg       0.96      0.92      0.94      2000
weighted avg       1.00      1.00      1.00      2000



In [8]:
# ============================================================
# CELL 6 – Quantum ML Fraud Detection (PennyLane VQC)
# ============================================================

N_QUBITS = 4
N_LAYERS = 2
dev = qml.device('default.qubit', wires=N_QUBITS)


@qml.qnode(dev, interface='autograd')
def vqc(inputs, weights):
    """
    Variational Quantum Classifier:
      1. Angle encoding via RY
      2. Ring-topology CNOT entanglement
      3. Trainable RY+RZ rotations per layer
      4. PauliZ expectation on qubit 0  → [-1, +1]
    Both inputs and weights MUST be pnp.ndarray for autograd.
    """
    for i in range(N_QUBITS):
        qml.RY(inputs[i], wires=i)

    for layer in range(N_LAYERS):
        for i in range(N_QUBITS - 1):
            qml.CNOT(wires=[i, i + 1])
        qml.CNOT(wires=[N_QUBITS - 1, 0])          # ring closure
        for i in range(N_QUBITS):
            qml.RY(weights[layer, i, 0], wires=i)
            qml.RZ(weights[layer, i, 1], wires=i)

    return qml.expval(qml.PauliZ(0))


def mse_loss(weights, X_batch, y_batch):
    """
    Batch MSE loss in {-1, +1} label space.
    weights must be pnp.array with requires_grad=True.
    """
    total = pnp.array(0.0, requires_grad=False)
    for x, y in zip(X_batch, y_batch):
        pred  = vqc(pnp.array(x, requires_grad=False), weights)
        label = pnp.array(1.0 if y == 1 else -1.0, requires_grad=False)
        total = total + (pred - label) ** 2
    return total / max(1, len(X_batch))


class QuantumFraudDetector:
    """Variational Quantum Classifier with explicit manual Adam training."""

    def __init__(self, n_qubits=N_QUBITS, n_layers=N_LAYERS,
                 lr=0.05, n_epochs=30, batch_size=32):
        self.n_qubits   = n_qubits
        self.n_layers   = n_layers
        self.n_epochs   = n_epochs
        self.batch_size = batch_size
        self.lr         = lr
        # Store as plain np.ndarray — cast to pnp only at gradient call boundaries
        self._weights_np = np.random.uniform(
            -np.pi, np.pi, (n_layers, n_qubits, 2)).astype(np.float64)
        self.loss_hist: list = []
        self.acc_hist:  list = []
        self.trained = False

    def _to_pnp(self):
        """Wrap stored weights as pnp array with grad tracking."""
        return pnp.array(self._weights_np, requires_grad=True)

    def _from_pnp(self, w):
        """Convert any pnp/autograd result back to plain np.ndarray."""
        return np.array(w, dtype=np.float64)

    def _predict_one(self, x):
        """Return fraud probability in [0, 1] for a single sample."""
        x_pnp = pnp.array(x, requires_grad=False)
        w_pnp = pnp.array(self._weights_np, requires_grad=False)
        raw   = float(vqc(x_pnp, w_pnp))   # in [-1, 1]
        return (raw + 1.0) / 2.0             # map to [0, 1]

    def fit(self, X_train, y_train):
        """Train on a balanced subsample using manual Adam (Colab-friendly)."""
        MAX = 300
        idx0 = np.where(y_train == 0)[0][:MAX // 2]
        idx1 = np.where(y_train == 1)[0][:MAX // 2]
        idx  = np.concatenate([idx0, idx1])
        np.random.shuffle(idx)
        Xq, yq = X_train[idx], y_train[idx]
        n = len(Xq)

        # Manual Adam state — all plain np.ndarray, no type surprises
        m  = np.zeros_like(self._weights_np)
        v  = np.zeros_like(self._weights_np)
        b1, b2, eps = 0.9, 0.999, 1e-8
        t  = 0

        grad_fn = qml.grad(mse_loss, argnum=0)

        print(f'⚛️  Training VQC on {n} samples × {self.n_epochs} epochs '
              f'(manual Adam, lr={self.lr})...')

        for epoch in range(self.n_epochs):
            perm = np.random.permutation(n)
            Xq, yq = Xq[perm], yq[perm]
            epoch_loss = 0.0
            n_batches  = 0

            for start in range(0, n, self.batch_size):
                Xb = Xq[start:start + self.batch_size]
                yb = yq[start:start + self.batch_size]

                # Gradient — cast to pnp here, convert result back immediately
                w_pnp = self._to_pnp()
                grad  = self._from_pnp(grad_fn(w_pnp, Xb, yb))

                # Loss for logging (no grad tracking needed)
                loss_val = float(mse_loss(
                    pnp.array(self._weights_np, requires_grad=False), Xb, yb))
                epoch_loss += loss_val
                n_batches  += 1

                # Adam parameter update (pure numpy — always ndarray)
                t  += 1
                m   = b1 * m + (1 - b1) * grad
                v   = b2 * v + (1 - b2) * grad ** 2
                m_h = m / (1 - b1 ** t)
                v_h = v / (1 - b2 ** t)
                self._weights_np -= self.lr * m_h / (np.sqrt(v_h) + eps)

            avg_loss = epoch_loss / max(1, n_batches)
            probs    = np.array([self._predict_one(x) for x in Xq])
            acc      = accuracy_score(yq, (probs > 0.5).astype(int))
            self.loss_hist.append(avg_loss)
            self.acc_hist.append(acc)

            if epoch % 5 == 0 or epoch == self.n_epochs - 1:
                print(f'  Epoch {epoch+1:02d}/{self.n_epochs}  '
                      f'Loss={avg_loss:.4f}  Acc={acc:.4f}')

        self.trained = True
        print('✅ VQC training complete!')

    def predict_proba(self, X):
        return np.array([self._predict_one(x) for x in X])

    def evaluate(self, X_test, y_test):
        probs = self.predict_proba(X_test)
        preds = (probs > 0.5).astype(int)
        acc   = accuracy_score(y_test, preds)
        auc   = roc_auc_score(y_test, probs)
        print(f'\n⚛️  Quantum VQC  Acc={acc:.4f}  AUC={auc:.4f}')
        print(classification_report(y_test, preds, target_names=['Normal', 'Fraud']))
        return acc, auc, probs

    def plot_training(self):
        fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
        a1.plot(self.loss_hist, 'b-o', ms=4); a1.set_title('VQC Training Loss')
        a1.set_xlabel('Epoch'); a1.set_ylabel('MSE Loss'); a1.grid(alpha=0.3)
        a2.plot(self.acc_hist,  'g-o', ms=4); a2.set_title('VQC Training Accuracy')
        a2.set_xlabel('Epoch'); a2.set_ylabel('Accuracy'); a2.grid(alpha=0.3)
        plt.tight_layout()
        return fig

    def draw_circuit(self):
        dummy_i = pnp.zeros(N_QUBITS, requires_grad=False)
        dummy_w = pnp.zeros((N_LAYERS, N_QUBITS, 2), requires_grad=False)
        fig, ax = qml.draw_mpl(vqc, expansion_strategy='device')(dummy_i, dummy_w)
        ax.set_title('VQC Circuit Diagram', fontsize=11)
        return fig


# ── Train ──────────────────────────────────────────────────────
quantum_detector = QuantumFraudDetector(n_epochs=30, lr=0.05)
quantum_detector.fit(X_q_train, y_train)

# ── Evaluate on balanced test subset ───────────────────────────
MAX_TEST = 100
idx0  = np.where(y_test == 0)[0][:MAX_TEST // 2]
idx1  = np.where(y_test == 1)[0][:MAX_TEST // 2]
idx_t = np.concatenate([idx0, idx1])
q_acc, q_auc, q_probs = quantum_detector.evaluate(X_q_test[idx_t], y_test[idx_t])

quantum_detector.plot_training(); plt.show()
quantum_detector.draw_circuit();  plt.show()

⚛️  Training VQC on 300 samples × 30 epochs (manual Adam, lr=0.05)...
  Epoch 01/30  Loss=1.0330  Acc=0.6000
  Epoch 06/30  Loss=0.9478  Acc=0.5767
  Epoch 11/30  Loss=0.8999  Acc=0.6400
  Epoch 16/30  Loss=0.8978  Acc=0.6833
  Epoch 21/30  Loss=0.8824  Acc=0.6033
  Epoch 26/30  Loss=0.8784  Acc=0.6833
  Epoch 30/30  Loss=0.8825  Acc=0.6267
✅ VQC training complete!

⚛️  Quantum VQC  Acc=0.6444  AUC=0.9540
              precision    recall  f1-score   support

      Normal       0.61      1.00      0.76        50
       Fraud       1.00      0.20      0.33        40

    accuracy                           0.64        90
   macro avg       0.80      0.60      0.55        90
weighted avg       0.78      0.64      0.57        90



In [9]:
# ============================================================
# CELL 7 – BB84 Quantum Key Distribution (Qiskit + Cirq)
# ============================================================

class BB84Protocol:
    """
    Full BB84 QKD simulation:
    - Alice prepares qubits; Bob measures
    - Optional Eve eavesdropping (50% intercept)
    - Sifting → secret key via second half of matched bits
    - QBER > 11%: eavesdropping detected
    """

    def __init__(self, n_bits=64, eve_present=False, eve_intercept_prob=0.5):
        self.n_bits          = n_bits
        self.eve_present     = eve_present
        self.eve_intercept_p = eve_intercept_prob
        self.backend         = AerSimulator()
        self.alice_bits = self.alice_bases = self.bob_bases = []
        self.bob_results = self.eve_bases = self.sifted_key = []
        self.error_rate = 0.0
        self.secret_key = self.key_hex = ''

    def _prepare(self, bit, basis):
        """basis 0=rectilinear, 1=diagonal."""
        qc = QuantumCircuit(1, 1)
        if bit == 1: qc.x(0)
        if basis == 1: qc.h(0)
        return qc

    def _measure(self, qc, basis):
        if basis == 1: qc.h(0)
        qc.measure(0, 0)
        return qc

    def run(self):
        random.seed(int(time.time()))
        self.alice_bits  = [random.randint(0,1) for _ in range(self.n_bits)]
        self.alice_bases = [random.randint(0,1) for _ in range(self.n_bits)]
        self.bob_bases   = [random.randint(0,1) for _ in range(self.n_bits)]
        self.eve_bases   = [random.randint(0,1) for _ in range(self.n_bits)]
        self.bob_results = []

        for i in range(self.n_bits):
            qc = self._prepare(self.alice_bits[i], self.alice_bases[i])

            # Eve intercepts with probability p
            if self.eve_present and random.random() < self.eve_intercept_p:
                qce = self._measure(qc.copy(), self.eve_bases[i])
                eve_bit = int(list(self.backend.run(qce, shots=1)
                                   .result().get_counts().keys())[0])
                qc = self._prepare(eve_bit, self.eve_bases[i])

            qc = self._measure(qc, self.bob_bases[i])
            result = int(list(self.backend.run(qc, shots=1)
                               .result().get_counts().keys())[0])
            self.bob_results.append(result)

        # Sifting
        match_idx = [i for i in range(self.n_bits)
                     if self.alice_bases[i] == self.bob_bases[i]]
        self.sifted_key = [self.alice_bits[i] for i in match_idx]
        bob_sifted      = [self.bob_results[i] for i in match_idx]

        # QBER estimation on first half
        check = max(1, len(self.sifted_key)//2)
        errs  = sum(a!=b for a,b in zip(self.sifted_key[:check], bob_sifted[:check]))
        self.error_rate = errs / check

        # Secret key from second half
        self.secret_key = ''.join(map(str, self.sifted_key[check:]))
        if len(self.secret_key) >= 8:
            bl = len(self.secret_key)//8
            self.key_hex = format(int(self.secret_key[:bl*8], 2), f'0{bl*2}x').upper()
        else:
            self.key_hex = 'INSUFFICIENT_BITS'

        return {
            'sifted_key_length'    : len(self.sifted_key),
            'secret_key_bits'      : self.secret_key,
            'secret_key_hex'       : self.key_hex,
            'error_rate'           : self.error_rate,
            'eavesdropping_detected': self.error_rate > 0.11,
        }

    def visualize(self):
        n = min(24, self.n_bits)
        fig, axes = plt.subplots(3, 1, figsize=(14, 8))
        fig.suptitle('BB84 Protocol – Step-by-Step', fontsize=13, fontweight='bold')

        ax = axes[0]
        clr = ['royalblue' if b==0 else 'crimson' for b in self.alice_bits[:n]]
        ax.bar(range(n), [1]*n, color=clr, edgecolor='white')
        ax.set_title('Alice Random Bits (blue=0, red=1)'); ax.set_yticks([])
        for i,(bit,basis) in enumerate(zip(self.alice_bits[:n],self.alice_bases[:n])):
            ax.text(i,.5,f'{bit}\n{"⊕" if basis else "+"}',ha='center',va='center',
                    fontsize=7,color='white',fontweight='bold')

        ax = axes[1]
        mclr = ['limegreen' if self.alice_bases[i]==self.bob_bases[i] else 'lightcoral'
                for i in range(n)]
        ax.bar(range(n),[1]*n,color=mclr,edgecolor='white')
        ax.set_title('Basis Match (green=keep, red=discard)'); ax.set_yticks([])
        for i in range(n):
            ax.text(i,.5,'✓' if self.alice_bases[i]==self.bob_bases[i] else '✗',
                    ha='center',va='center',fontsize=10,fontweight='bold')

        ax = axes[2]
        kbits = [int(b) for b in list(self.secret_key[:n])]
        ax.bar(range(len(kbits)), kbits, color='mediumpurple', edgecolor='white')
        ax.set_title(f'Secret Key Bits | QBER={self.error_rate:.2%} | Hex: {self.key_hex[:16]}...')
        ax.set_xlabel('Bit index'); ax.set_ylabel('Bit'); ax.grid(axis='y',alpha=0.3)

        plt.tight_layout(); return fig

    def cirq_representation(self):
        q = cirq.LineQubit.range(2)
        circuit = cirq.Circuit([
            cirq.H(q[0]),
            cirq.CNOT(q[0], q[1]),
            cirq.measure(*q, key='result')
        ])
        return str(circuit)


# Run without Eve
print('🔑 BB84 without eavesdropper:')
bb84_clean = BB84Protocol(n_bits=64, eve_present=False)
r_clean = bb84_clean.run()
print(f'  Sifted bits: {r_clean["sifted_key_length"]}  QBER: {r_clean["error_rate"]:.2%}')
print(f'  Key (hex)  : {r_clean["secret_key_hex"]}')

# Run with Eve
print('\n🕵️  BB84 with eavesdropper:')
bb84_eve = BB84Protocol(n_bits=64, eve_present=True, eve_intercept_prob=0.8)
r_eve = bb84_eve.run()
print(f'  QBER: {r_eve["error_rate"]:.2%}  Detected: {r_eve["eavesdropping_detected"]}')

print('\n⚛️  Cirq circuit representation:')
print(bb84_clean.cirq_representation())

bb84_clean.visualize(); plt.show()


🔑 BB84 without eavesdropper:
  Sifted bits: 30  QBER: 0.00%
  Key (hex)  : 44

🕵️  BB84 with eavesdropper:
  QBER: 6.67%  Detected: False

⚛️  Cirq circuit representation:
0: ───H───@───M('result')───
          │   │
1: ───────X───M─────────────


In [10]:
# ============================================================
# CELL 8 – Secure Payment Authentication Logic
# ============================================================

class SecurePaymentAuthenticator:
    """
    Orchestrates the full quantum-secure payment flow:
      1. Feature engineering from payment details
      2. Classical + quantum fraud scoring
      3. BB84 quantum key generation
      4. Decision (APPROVED / REVIEW / BLOCKED)
    """

    HIGH_RISK_MERCHANTS = {'casino','crypto','gambling','forex','betting'}

    def __init__(self, classical, quantum, q_scaler, q_cols):
        self.classical = classical
        self.quantum   = quantum
        self.q_scaler  = q_scaler
        self.q_cols    = q_cols
        self.log: List[Dict] = []

    def _engineer_features(self, amount, merchant, location, hour, card_suffix):
        """
        Map payment metadata → 30-dimensional feature vector
        matching the trained model input shape.
        """
        seed = int(hashlib.md5(f'{amount}{merchant}{location}'.encode()).hexdigest(), 16) % 2**32
        rng  = np.random.RandomState(seed)
        feats = rng.normal(0, 1, 28)   # V1..V28

        # Inject fraud signals based on heuristics
        if any(h in merchant.lower() for h in self.HIGH_RISK_MERCHANTS):
            feats[[3,9,10]] += rng.uniform(1.5, 3.0, 3)
        if hour < 4 or hour > 22:
            feats[[11,14]] += rng.uniform(1.0, 2.0, 2)
        if amount > 1000:
            feats[[4,12]] += rng.uniform(0.5, 2.5, 2)

        amount_s = (np.log1p(amount) - 4.0) / 2.0
        time_s   = (hour / 24.0 - 0.5) * 2.0
        return np.concatenate([feats, [amount_s, time_s]])   # shape (30,)

    def _classical_score(self, features):
        return self.classical.predict_proba_rf(features)

    def _quantum_score(self, features):
        if not self.quantum.trained:
            return 0.5
        q = self.q_scaler.transform(features[self.q_cols].reshape(1,-1))
        return float(self.quantum.predict_proba(q)[0])

    def _combine(self, c, q, wc=0.6, wq=0.4):
        return wc*c + wq*q

    @staticmethod
    def _risk(prob):
        return 'LOW' if prob < 0.3 else ('MEDIUM' if prob < 0.6 else 'HIGH')

    @staticmethod
    def _decide(prob):
        if prob < 0.40: return '✅ APPROVED', 'Transaction approved by quantum-secure system.'
        if prob < 0.70: return '⚠️ REVIEW',   'Flagged for additional verification.'
        return '🚫 BLOCKED', 'High fraud probability – transaction blocked.'

    def authenticate(self, card_number, amount, merchant, location, time_str, eve_present=False):
        t0 = time.time()
        try:
            hour = int(time_str.split(':')[0])
        except Exception:
            hour = 12
        card_suffix = (card_number.replace(' ',''))[-4:]

        feats          = self._engineer_features(amount, merchant, location, hour, card_suffix)
        c_prob         = self._classical_score(feats)
        q_prob         = self._quantum_score(feats)
        combined       = self._combine(c_prob, q_prob)

        bb84           = BB84Protocol(n_bits=64, eve_present=eve_present)
        key_res        = bb84.run()

        decision, reason = self._decide(combined)
        result = {
            'timestamp'           : datetime.now().isoformat(),
            'card_suffix'         : card_suffix,
            'amount'              : amount,
            'merchant'            : merchant,
            'location'            : location,
            'hour'                : hour,
            'classical_fraud_prob': round(c_prob, 4),
            'quantum_fraud_prob'  : round(q_prob, 4),
            'combined_fraud_prob' : round(combined, 4),
            'risk_level'          : self._risk(combined),
            'decision'            : decision,
            'reason'              : reason,
            'quantum_key_hex'     : key_res['secret_key_hex'],
            'quantum_key_bits'    : key_res['secret_key_bits'][:32],
            'qber'                : round(key_res['error_rate'], 4),
            'eavesdrop_detected'  : key_res['eavesdropping_detected'],
            'channel_secure'      : not key_res['eavesdropping_detected'],
            'processing_ms'       : round((time.time()-t0)*1000, 1),
            '_features'           : feats,   # kept for LR prob computation in UI
        }
        self.log.append(result)
        return result


authenticator = SecurePaymentAuthenticator(
    classical=classical_detector, quantum=quantum_detector,
    q_scaler=q_scaler, q_cols=q_cols)

# Smoke test
print('🔐 Smoke test:')
r = authenticator.authenticate('4111 1111 1111 1234', 250.0, 'Amazon',
                               'New York, US', '14:30', eve_present=False)
for k,v in r.items():
    if k != '_features':
        print(f'  {k:<28}: {v}')


🔐 Smoke test:
  timestamp                   : 2026-03-08T12:33:47.824363
  card_suffix                 : 1234
  amount                      : 250.0
  merchant                    : Amazon
  location                    : New York, US
  hour                        : 14
  classical_fraud_prob        : 0.0
  quantum_fraud_prob          : 0.3622
  combined_fraud_prob         : 0.1449
  risk_level                  : LOW
  decision                    : ✅ APPROVED
  reason                      : Transaction approved by quantum-secure system.
  quantum_key_hex             : A1
  quantum_key_bits            : 10100001101000
  qber                        : 0.0
  eavesdrop_detected          : False
  channel_secure              : True
  processing_ms               : 101.0


In [11]:
# ============================================================
# CELL 9 – Visualization Helpers
# ============================================================

def plot_fraud_gauge(prob):
    """Semi-circular risk gauge."""
    fig, ax = plt.subplots(figsize=(5, 3), subplot_kw={'aspect':'equal'})
    theta = np.linspace(np.pi, 0, 300)
    ax.fill_between(np.cos(theta), np.sin(theta), 0.8*np.sin(theta), color='#e0e0e0')
    th_p  = np.linspace(np.pi, np.pi-np.pi*prob, 300)
    color = '#2ecc71' if prob<0.4 else ('#f39c12' if prob<0.7 else '#e74c3c')
    ax.fill_between(np.cos(th_p), np.sin(th_p), 0.8*np.sin(th_p), color=color, alpha=0.9)
    angle = np.pi - np.pi*prob
    ax.annotate('', xy=(0.7*np.cos(angle), 0.7*np.sin(angle)), xytext=(0,0),
                arrowprops=dict(arrowstyle='->', color='black', lw=2))
    ax.text(0,-0.15, f'{prob:.1%}', ha='center', fontsize=16, fontweight='bold', color=color)
    ax.text(0,-0.35, 'Fraud Probability', ha='center', fontsize=9, color='#555')
    ax.text(-0.95, 0.05, 'SAFE', ha='center', fontsize=8, color='#2ecc71')
    ax.text( 0.90, 0.05, 'RISK', ha='center', fontsize=8, color='#e74c3c')
    ax.set_xlim(-1.1,1.1); ax.set_ylim(-0.5,1.1); ax.axis('off')
    ax.set_title('Fraud Risk Gauge', fontsize=10)
    return fig


def plot_model_comparison(lr_prob, rf_prob, q_prob, combined):
    """Bar chart comparing model fraud probabilities."""
    labels = ['Logistic\nRegression','Random\nForest','Quantum\nVQC','Combined']
    vals   = [lr_prob, rf_prob, q_prob, combined]
    colors = ['#3498db','#2ecc71','#9b59b6','#e74c3c']
    fig, ax = plt.subplots(figsize=(7,4))
    bars = ax.bar(labels, vals, color=colors, edgecolor='white', lw=1.5)
    ax.axhline(0.5, color='gray', ls='--', lw=1, label='Decision threshold')
    ax.set_ylim(0,1); ax.set_ylabel('Fraud Probability')
    ax.set_title('Model Comparison – Fraud Probability')
    ax.legend(fontsize=8); ax.grid(axis='y', alpha=0.3)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02,
                f'{val:.2%}', ha='center', fontsize=9, fontweight='bold')
    plt.tight_layout(); return fig


def plot_bb84_summary(qber, key_hex, eavesdrop):
    """Visual summary card for BB84 results."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
    fig.suptitle('BB84 Quantum Key Distribution Results', fontsize=12, fontweight='bold')

    qc = '#e74c3c' if qber > 0.11 else '#2ecc71'
    ax1.barh(['QBER'], [qber], color=qc, height=0.4)
    ax1.barh(['QBER'], [1-qber], left=[qber], color='#e0e0e0', height=0.4)
    ax1.axvline(0.11, color='orange', ls='--', lw=2, label='Threshold 11%')
    ax1.set_xlim(0,1); ax1.set_title('Quantum Bit Error Rate'); ax1.legend(fontsize=8)
    ax1.text(qber/2, 0, f'{qber:.1%}', ha='center', va='center',
             fontsize=12, fontweight='bold', color='white')

    ax2.axis('off')
    sc = '#e74c3c' if eavesdrop else '#2ecc71'
    st = '⚠️ EAVESDROPPER DETECTED' if eavesdrop else '✅ CHANNEL SECURE'
    ax2.text(0.5, 0.80, st, ha='center', fontsize=13, color=sc,
             fontweight='bold', transform=ax2.transAxes)
    ax2.text(0.5, 0.55, 'Quantum Key (hex):', ha='center', fontsize=9,
             color='#666', transform=ax2.transAxes)
    disp_key = '\n'.join(key_hex[i:i+16] for i in range(0, min(32,len(key_hex)), 16))
    ax2.text(0.5, 0.30, disp_key, ha='center', fontsize=10, fontweight='bold',
             color='#2c3e50', fontfamily='monospace', transform=ax2.transAxes)
    plt.tight_layout(); return fig


print('✅ Visualization helpers ready.')
# Quick preview
fig = plot_fraud_gauge(0.72)
plt.show()


✅ Visualization helpers ready.


In [12]:
# ============================================================
# CELL 10 – Gradio UI Handler Functions
# ============================================================

TXN_HISTORY: List[Dict] = []


def authenticate_payment(card_number, amount, merchant, location,
                         txn_time, eve_present):
    """Main Gradio callback."""

    # Validate inputs
    if not card_number or len(card_number.replace(' ','')) < 12:
        return ('⚠️ Invalid card number',) + ('',)*8 + (None,)*3
    if not amount or amount <= 0:
        return ('⚠️ Amount must be positive',) + ('',)*8 + (None,)*3

    try:
        result = authenticator.authenticate(
            card_number=card_number.replace(' ',''),
            amount=float(amount), merchant=merchant,
            location=location, time_str=txn_time,
            eve_present=bool(eve_present))

        TXN_HISTORY.append(result)

        # Get LR probability for comparison chart
        feats  = result.get('_features', np.zeros(30))
        lr_prob = classical_detector.predict_proba_lr(feats)
        rf_prob = result['classical_fraud_prob']
        q_prob  = result['quantum_fraud_prob']
        comb    = result['combined_fraud_prob']

        key_disp = (result['quantum_key_hex'][:32]+'...'
                    if len(result['quantum_key_hex'])>32
                    else result['quantum_key_hex'])

        channel_disp = ('✅ SECURE'
                        if result['channel_secure']
                        else '🚨 COMPROMISED – Eve detected!')

        summary = f"""
### 🔐 Authentication Report
| Field    | Value |
|----------|-------|
| Card     | ****-****-****-{result['card_suffix']} |
| Amount   | ${result['amount']:,.2f} |
| Merchant | {merchant} |
| Location | {location} |
| Time     | {txn_time} |

**Decision: {result['decision']}**
{result['reason']}

| Model | Fraud Prob |
|-------|------------|
| Logistic Regression | {lr_prob:.2%} |
| Random Forest | {rf_prob:.2%} |
| Quantum VQC | {q_prob:.2%} |
| **Combined** | **{comb:.2%}** |

**Quantum Key:** `{key_disp}`
**QBER:** {result['qber']:.2%}
**Channel:** {channel_disp}
**Processing:** {result['processing_ms']} ms
"""
        fig_g  = plot_fraud_gauge(comb)
        fig_c  = plot_model_comparison(lr_prob, rf_prob, q_prob, comb)
        fig_bb = plot_bb84_summary(result['qber'], result['quantum_key_hex'],
                                   result['eavesdrop_detected'])

        return (
            result['decision'],
            result['reason'],
            f"{comb:.1%}",
            result['risk_level'],
            key_disp,
            f"{result['qber']:.2%}",
            channel_disp,
            f"{result['processing_ms']} ms",
            summary,
            fig_g, fig_c, fig_bb,
        )
    except Exception as e:
        import traceback
        err = f'❌ Error: {e}\n{traceback.format_exc()}'
        return (err,) + ('',)*8 + (None,)*3


def show_history():
    if not TXN_HISTORY:
        return '_No transactions yet. Submit one above!_'
    header = '| Time | Card | Amount | Merchant | Fraud% | Decision |\n'
    header += '|------|------|--------|----------|--------|----------|\n'
    rows = []
    for t in TXN_HISTORY[-10:]:
        rows.append(
            f"| {t['timestamp'][-8:]} | ****{t['card_suffix']} |"
            f" ${t['amount']:.2f} | {t['merchant'][:18]} |"
            f" {t['combined_fraud_prob']:.1%} | {t['decision'].split()[0]} |")
    return header + '\n'.join(rows)


print('✅ Gradio handler functions ready.')


✅ Gradio handler functions ready.


In [13]:
# ============================================================
# CELL 11 – Launch Gradio Application
# ============================================================
# This is the main interactive UI.

THEME = gr.themes.Soft(primary_hue='violet', secondary_hue='blue', neutral_hue='slate')

with gr.Blocks(theme=THEME, title='⚛️ Quantum Secure Payment Auth') as demo:

    gr.Markdown("""
    # 🔐 Quantum Secure Payment Authentication System
    ### BB84 QKD · Variational Quantum Classifier · Fraud Detection
    > Enter payment details and click **Check Transaction** to run the full
    > quantum-secure authentication pipeline.
    """)

    with gr.Tabs():

        # ── Tab 1: Payment ─────────────────────────────────────
        with gr.Tab('💳 Payment Authentication'):
            with gr.Row():

                with gr.Column(scale=1):
                    gr.Markdown('### 📝 Transaction Details')

                    card_in     = gr.Textbox(label='Card Number',
                                             placeholder='4111 1111 1111 1234',
                                             value='4111 1111 1111 1234', max_lines=1)
                    amount_in   = gr.Number(label='Amount (USD)', value=250.00,
                                            minimum=0.01, maximum=100_000)
                    merchant_in = gr.Dropdown(
                        label='Merchant',
                        choices=['Amazon','Walmart','Apple Store','Netflix','Uber',
                                 'Airbnb','Crypto Exchange','Online Casino',
                                 'International Transfer','Local Restaurant','Gas Station'],
                        value='Amazon', allow_custom_value=True)
                    location_in = gr.Dropdown(
                        label='Location',
                        choices=['New York, US','London, UK','Tokyo, JP','Sydney, AU',
                                 'Lagos, NG','Moscow, RU','São Paulo, BR',
                                 'Dubai, AE','Unknown'],
                        value='New York, US', allow_custom_value=True)
                    time_in     = gr.Textbox(label='Transaction Time (HH:MM)',
                                             value='14:30', max_lines=1)
                    eve_in      = gr.Checkbox(label='🕵️ Simulate Eavesdropper (Eve)',
                                              value=False,
                                              info='Tests BB84 eavesdropping detection.')
                    submit_btn  = gr.Button('🔐 Check Transaction',
                                            variant='primary', size='lg')

                with gr.Column(scale=1):
                    gr.Markdown('### 📊 Authentication Results')
                    decision_out = gr.Textbox(label='Decision',          interactive=False)
                    reason_out   = gr.Textbox(label='Reason',            interactive=False)
                    fraud_out    = gr.Textbox(label='Fraud Probability', interactive=False)
                    risk_out     = gr.Textbox(label='Risk Level',        interactive=False)
                    key_out      = gr.Textbox(label='Quantum Key (hex)', interactive=False)
                    qber_out     = gr.Textbox(label='QBER',              interactive=False)
                    channel_out  = gr.Textbox(label='Channel Security',  interactive=False)
                    proc_out     = gr.Textbox(label='Processing Time',   interactive=False)

            summary_out  = gr.Markdown()
            gr.Markdown('### 📈 Visualizations')
            with gr.Row():
                gauge_plt   = gr.Plot(label='Fraud Risk Gauge')
                compare_plt = gr.Plot(label='Model Comparison')
            bb84_plt = gr.Plot(label='BB84 Key Generation')

        # ── Tab 2: History ─────────────────────────────────────
        with gr.Tab('📜 Transaction History'):
            hist_btn = gr.Button('🔄 Refresh')
            hist_out = gr.Markdown()
            hist_btn.click(show_history, outputs=hist_out)

        # ── Tab 3: System Info ─────────────────────────────────
        with gr.Tab('ℹ️ System Info'):
            lr_auc = classical_detector.metrics.get('Logistic Regression',{}).get('auc',0)
            rf_auc = classical_detector.metrics.get('Random Forest',{}).get('auc',0)
            gr.Markdown(f"""
## Architecture Overview

### 🔬 Quantum Components
| Component | Library | Role |
|-----------|---------|------|
| Variational Quantum Classifier | PennyLane | Fraud detection |
| BB84 Protocol | Qiskit + Aer | Quantum key distribution |
| Circuit Visualization | Cirq | Qubit state representation |

### 🤖 Classical ML
| Model | AUC |
|-------|-----|
| Logistic Regression | {lr_auc:.4f} |
| Random Forest | {rf_auc:.4f} |
| Quantum VQC | {q_auc:.4f} |

### ⚖️ Decision Thresholds
| Score | Decision |
|-------|----------|
| < 40% | ✅ APPROVED |
| 40–70% | ⚠️ REVIEW |
| > 70% | 🚫 BLOCKED |

### 🔑 BB84 Parameters
- Qubits per session: **64**
- QBER eavesdropping threshold: **11%**
- Key format: **Hexadecimal**
- Combined score weights: Classical **60%** + Quantum **40%**
            """)

    # Wire submit button
    submit_btn.click(
        fn=authenticate_payment,
        inputs=[card_in, amount_in, merchant_in, location_in, time_in, eve_in],
        outputs=[decision_out, reason_out, fraud_out, risk_out,
                 key_out, qber_out, channel_out, proc_out,
                 summary_out, gauge_plt, compare_plt, bb84_plt],
    )
    demo.load(show_history, outputs=hist_out)

print('✅ Gradio app built – launching...')
demo.launch(share=True, debug=False, show_error=True, quiet=False)


✅ Gradio app built – launching...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://101b19071fcd7c8845.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [14]:
# ============================================================
# CELL 12 – Demo Interactions & Summary Analytics
# ============================================================
# Run 5 representative transactions and display an analytics dashboard.

DEMO_TXN = [
    dict(card_number='4111111111111111', amount=75.00,
         merchant='Local Restaurant', location='New York, US',
         time_str='19:15', eve_present=False),

    dict(card_number='5500000000000004', amount=4500.00,
         merchant='Crypto Exchange', location='Unknown',
         time_str='02:47', eve_present=False),

    dict(card_number='4000000000000002', amount=299.99,
         merchant='Apple Store', location='London, UK',
         time_str='11:30', eve_present=False),

    dict(card_number='4111111111111234', amount=18000.00,
         merchant='Online Casino', location='Lagos, NG',
         time_str='03:15', eve_present=True),

    dict(card_number='6011111111111117', amount=12.50,
         merchant='Netflix', location='Tokyo, JP',
         time_str='20:00', eve_present=False),
]

print('🚀 Running demo transactions...\n')
demo_results = []
for i, txn in enumerate(DEMO_TXN, 1):
    res = authenticator.authenticate(**txn)
    demo_results.append(res)
    print(f'  Txn {i}: ${txn["amount"]:>8.2f}  {txn["merchant"]:<22}  '
          f'Fraud={res["combined_fraud_prob"]:.1%}  '
          f'QBER={res["qber"]:.1%}  '
          f'{res["decision"]}')

# ── Summary Analytics Dashboard ──────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Demo Transaction Analytics Dashboard',
             fontsize=14, fontweight='bold')

merchants   = [r['merchant']            for r in demo_results]
fraud_probs = [r['combined_fraud_prob'] for r in demo_results]
qbers       = [r['qber']                for r in demo_results]
decisions   = [r['decision'].split()[0] for r in demo_results]
amounts     = [r['amount']              for r in demo_results]

# 1 – Fraud probability per txn
ax = axes[0,0]
clr = ['#e74c3c' if p>0.6 else ('#f39c12' if p>0.3 else '#2ecc71') for p in fraud_probs]
ax.bar(range(len(merchants)), fraud_probs, color=clr, edgecolor='white')
ax.axhline(0.4, color='orange', ls='--', lw=1.5, label='Low threshold')
ax.axhline(0.7, color='red',    ls='--', lw=1.5, label='High threshold')
ax.set_xticks(range(len(merchants)))
ax.set_xticklabels([m[:14] for m in merchants], rotation=20, ha='right')
ax.set_ylabel('Fraud Probability')
ax.set_title('Fraud Probability by Transaction')
ax.legend(fontsize=7); ax.grid(axis='y', alpha=0.3)

# 2 – QBER per txn
ax = axes[0,1]
ax.bar(range(len(merchants)), qbers,
       color=['#e74c3c' if q>0.11 else '#2ecc71' for q in qbers],
       edgecolor='white')
ax.axhline(0.11, color='red', ls='--', lw=1.5, label='QBER threshold')
ax.set_xticks(range(len(merchants)))
ax.set_xticklabels([m[:14] for m in merchants], rotation=20, ha='right')
ax.set_ylabel('QBER'); ax.set_title('Quantum Bit Error Rate')
ax.legend(fontsize=7); ax.grid(axis='y', alpha=0.3)

# 3 – Decision pie
ax = axes[1,0]
dec_counts = Counter(decisions)
ax.pie(dec_counts.values(), labels=list(dec_counts.keys()),
       colors=['#2ecc71','#f39c12','#e74c3c'][:len(dec_counts)],
       autopct='%1.0f%%', startangle=90)
ax.set_title('Transaction Decision Distribution')

# 4 – Amount vs Fraud scatter
ax = axes[1,1]
sc = ['#e74c3c' if p>0.5 else '#2ecc71' for p in fraud_probs]
ax.scatter(amounts, fraud_probs, c=sc, s=150, edgecolors='white', zorder=3)
for i,m in enumerate(merchants):
    ax.annotate(m[:12], (amounts[i], fraud_probs[i]),
                textcoords='offset points', xytext=(5,5), fontsize=7)
ax.set_xlabel('Amount (USD)'); ax.set_ylabel('Fraud Probability')
ax.set_title('Amount vs Fraud Probability'); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('demo_dashboard.png', dpi=120, bbox_inches='tight')
plt.show()

print('\n📋 Final Model Performance Summary:')
print(f'  Logistic Regression AUC : {classical_detector.metrics.get("Logistic Regression",{}).get("auc",0):.4f}')
print(f'  Random Forest AUC       : {classical_detector.metrics.get("Random Forest",{}).get("auc",0):.4f}')
print(f'  Quantum VQC AUC         : {q_auc:.4f}')
print(f'  Quantum VQC Accuracy    : {q_acc:.4f}')
print('\n🔐 System ready! The Gradio UI above is live and accepting transactions.')
print('   Open the share link from Cell 11 to access the full interactive interface.')


🚀 Running demo transactions...

  Txn 1: $   75.00  Local Restaurant        Fraud=15.4%  QBER=0.0%  ✅ APPROVED
  Txn 2: $ 4500.00  Crypto Exchange         Fraud=27.5%  QBER=0.0%  ✅ APPROVED
  Txn 3: $  299.99  Apple Store             Fraud=15.0%  QBER=0.0%  ✅ APPROVED
  Txn 4: $18000.00  Online Casino           Fraud=40.0%  QBER=10.5%  ⚠️ REVIEW
  Txn 5: $   12.50  Netflix                 Fraud=14.8%  QBER=0.0%  ✅ APPROVED

📋 Final Model Performance Summary:
  Logistic Regression AUC : 0.6018
  Random Forest AUC       : 0.9990
  Quantum VQC AUC         : 0.9540
  Quantum VQC Accuracy    : 0.6444

🔐 System ready! The Gradio UI above is live and accepting transactions.
   Open the share link from Cell 11 to access the full interactive interface.
